# **Homework Assignment QT3: Quantifying mRNA Abundance with Kallisto**

**Name:** Vedasravas Dasari

**M ID:** M16446787

**Colab Link:** https://colab.research.google.com/drive/1u76iafLLP9-JYVbtgIUWpZ8hUtdo9G94?usp=sharing

**Submission Date:** 03/04/2025

---

**Course:** CS7099-Introduction to Bioinformatics

**Instructor:** Jarek Meller

---
All results, figures, and discussion are included below.

Step 1: Install Kallisto and play with the toy example while testing the effect of kmer length.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
# Install Kallisto
!apt-get -qq install kallisto
# Download also the distribution below to extract a toy example for testing
!wget https://github.com/pachterlab/kallisto/releases/download/v0.46.1/kallisto_linux-v0.46.1.tar.gz
!gunzip -f kallisto_linux-v0.46.1.tar.gz
!tar -xvf kallisto_linux-v0.46.1.tar
#!ls -al kallisto/test/*
# Index the transcriptome defined in transcripts.fasta.gz file
!kallisto index -i my_transcripts.idx -k 29 kallisto/test/transcripts.fasta.gz
# Run pseudo-alignment based quantification
!kallisto quant -i my_transcripts.idx -o output kallisto/test/reads_1.fastq.gz kallisto/test/reads_2.fastq.gz
# Check the output
!head -10 output/abundance.tsv
!cp output/abundance.tsv /content/drive/MyDrive/.


Mounted at /content/drive
Selecting previously unselected package libhtscodecs2:amd64.
(Reading database ... 124926 files and directories currently installed.)
Preparing to unpack .../libhtscodecs2_1.1.1-3_amd64.deb ...
Unpacking libhtscodecs2:amd64 (1.1.1-3) ...
Selecting previously unselected package libhts3:amd64.
Preparing to unpack .../libhts3_1.13+ds-2build1_amd64.deb ...
Unpacking libhts3:amd64 (1.13+ds-2build1) ...
Selecting previously unselected package kallisto.
Preparing to unpack .../kallisto_0.46.2+dfsg-3_amd64.deb ...
Unpacking kallisto (0.46.2+dfsg-3) ...
Setting up libhtscodecs2:amd64 (1.1.1-3) ...
Setting up libhts3:amd64 (1.13+ds-2build1) ...
Setting up kallisto (0.46.2+dfsg-3) ...
Processing triggers for man-db (2.10.2-1) ...
Processing triggers for libc-bin (2.35-0ubuntu3.8) ...
/sbin/ldconfig.real: /usr/local/lib/libtbbbind_2_0.so.3 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtcm_debug.so.1 is not a symbolic link

/sbin/ldconfig.real: /usr/local/

Part A: Using the code in Step 1 above, generate index files for kmer length k=27, 29, 31 and compare the estimated abundance values in the last column (TPM). Are the results strongly dependent on the choice of k?

Part B: This time use the actual human transcriptome and process real RNA-seq samples, Follow the steps below.

Step 2: Get the RefSeq and Ensembl version of the human transcriptome. Index both using Kallisto (please note this may take some time) and compare the results. Which do you think is better and why?

In [ ]:
#
# Downlaod and index human reference transcriptomes from RefSeq and Ensembl.
#
!wget ftp.ncbi.nlm.nih.gov/refseq/H_sapiens/annotation/GRCh38_latest/refseq_identifiers/GRCh38_latest_rna.fna.gz
!wget ftp.ensembl.org/pub/current_fasta/homo_sapiens/cdna/Homo_sapiens.GRCh38.cdna.all.fa.gz
# Comment/uncomment as needed to test
#!kallisto index -i refseq_transcripts.idx -k 31 GRCh38_latest_rna.fna.gz
!kallisto index -i ensembl_transcripts.idx -k 31 Homo_sapiens.GRCh38.cdna.all.fa.gz
# note that indexing the human transcriptome may take about 10 minutes

--2025-02-20 02:38:53--  http://ftp.ncbi.nlm.nih.gov/refseq/H_sapiens/annotation/GRCh38_latest/refseq_identifiers/GRCh38_latest_rna.fna.gz
Resolving ftp.ncbi.nlm.nih.gov (ftp.ncbi.nlm.nih.gov)... 130.14.250.31, 130.14.250.10, 130.14.250.11, ...
Connecting to ftp.ncbi.nlm.nih.gov (ftp.ncbi.nlm.nih.gov)|130.14.250.31|:80... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://ftp.ncbi.nlm.nih.gov/refseq/H_sapiens/annotation/GRCh38_latest/refseq_identifiers/GRCh38_latest_rna.fna.gz [following]
--2025-02-20 02:38:54--  https://ftp.ncbi.nlm.nih.gov/refseq/H_sapiens/annotation/GRCh38_latest/refseq_identifiers/GRCh38_latest_rna.fna.gz
Connecting to ftp.ncbi.nlm.nih.gov (ftp.ncbi.nlm.nih.gov)|130.14.250.31|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 135018606 (129M) [application/x-gzip]
Saving to: ‘GRCh38_latest_rna.fna.gz’

GRCh38_latest_rna.f 100%[===================>] 128.76M  10.5MB/s    in 14s     

2025-02-20 02:39:08 (9

Step 3: Install BioMart and define a function to extract EnsemblID - Gene Symble table to convert Ensembl transcript IDs into gene symbols using BioMart.

This little bit of magic from Ben Heil will come in handy later. For an interactive conversion of RefSeq ID or Ensembl ID to gene names you can use https://www.biotools.fr/human/refseq_symbol_converter or Biomart at Ensembl.

In [ ]:
!pip install biomart

import biomart

def get_ensembl_mappings():
    # Set up connection to the server
    server = biomart.BiomartServer('http://useast.ensembl.org/biomart')
    # Define the target genome
    mart = server.datasets['hsapiens_gene_ensembl']

    # List the types of data we could use
    attributes = ['ensembl_transcript_id', 'external_gene_name',
                  'ensembl_gene_id', 'ensembl_peptide_id']

    # Get the mapping between the attributes
    response = mart.search({'attributes': attributes})
    data = response.raw.data.decode('ascii')

    ensembl_to_genesymbol = {}
    # Store the data in a dictionary
    for line in data.splitlines():
        line = line.split('\t')
        # The entries are in the same order as in the `attributes` variable
        transcript_id = line[0]
        gene_symbol = line[1]
        ensembl_gene = line[2]
        ensembl_peptide = line[3]

        # Note that some of these keys may be an empty string
        ensembl_to_genesymbol[transcript_id] = gene_symbol
        ensembl_to_genesymbol[ensembl_gene] = gene_symbol
        ensembl_to_genesymbol[ensembl_peptide] = gene_symbol

    return ensembl_to_genesymbol

# Call and test it here
ensembl_to_genesymbol = get_ensembl_mappings()
transcript_id = "ENST00000455134"
print(ensembl_to_genesymbol[transcript_id])


GAPDHP28


Step 4: Define a little function to use the big function above to convert Kallisto output into something a human can understand and use.

In [ ]:
#
# Define a little function to extract TPM values from Kallisto output
# Substitute Ensembl transcript IDs ENST... into official gene names
#
def get_gene_tpm_table(output_folder,tpm_threshold):
  output_file = output_folder + "/abundance.tsv"
  output_file_short = output_folder + "/gene_tpm.tsv"
  f=open(output_file)
  f_out=open(output_file_short,"w")
  lines=f.readlines()[1:]
  highly_abundant_transcripts = []
  for transcript_line in lines:
    my_precious = transcript_line.split()[-1]
    transcript_id = transcript_line.split()[0]
    transcript_id = transcript_id.split('.')[0]
    if(float(my_precious) > tpm_threshold):
      #print(transcript_line)
      highly_abundant_transcripts.append(transcript_id)
      print(ensembl_to_genesymbol[transcript_id],my_precious,file=f_out)
  f_out.close()
  #!ls -al $output_folder
  !head -10 $output_file_short
  !grep "ESR1" $output_file_short
  return highly_abundant_transcripts

Step 5: Download a subset of 5 LumA and 4 Basal breast tumor samples from a recent study that you can find here: https://www.ncbi.nlm.nih.gov/bioproject/PRJNA454288.

Note that the SRA ID for this project is SRP144041.

Use this link to modify the selection of files from SRA:
https://www.ncbi.nlm.nih.gov/Traces/study/?acc=SRP144041&f=pam50_subtype_sam_s%3An%3Aluma%2Cbasal%3Binstrument_s%3An%3Aillumina%2520miseq%3Brun_file_create_date_dt%3An%3A2018-04-30T14%253A02%253A00Z%3Ac&o=acc_s%3Aa&s=SRR7083635,SRR7083641,SRR7083648,SRR7083650,SRR7083639,SRR7083644,SRR7083649,SRR7083651,SRR7083652

You can also download metadata, including LumA vs. Basal subtype for sample there. More on downloading files from SRA: https://bioinformaticsworkbook.org/dataAcquisition/fileTransfer/sra.html#gsc.tab=0

In [ ]:
# Get some real data to play with
#
# Using SRP144041 project with a breast cancer cohort limited to 9 LumA and Basal samples
#
accessions = ["SRR7083635", "SRR7083641", "SRR7083648", "SRR7083650", "SRR7083639", "SRR7083644", "SRR7083649", "SRR7083651", "SRR7083652"]
# For tests, use the first (or the last) three fastq files
#accessions = ["SRR7083635", "SRR7083641", "SRR7083648"]
accessions = ["SRR7083649", "SRR7083651", "SRR7083652"]
# Set the threshold for abundance in terms of TPMs: ignore genes with lower expression
tpm_threshold = 100.0
path2fastq = "https://trace.ncbi.nlm.nih.gov/Traces/sra-reads-be/fastq?acc="
for sra_accession in accessions:
  SRA_fastq_file = path2fastq + sra_accession
  fastq_file = sra_accession + ".fastq.gz"
  output_folder = "output_" + sra_accession
  print("\n Downloading and processing: ",fastq_file,"\n")
  # Download a fastq file from the list
  !wget -O $fastq_file $SRA_fastq_file
  # Run Kallisto to quantify transcripts
  !kallisto quant -i ensembl_transcripts.idx --single -l 75.0 -s 5.0 -o $output_folder $fastq_file
  # Simplify the output and report TPMs using gene names
  highly_abundant_transcripts = get_gene_tpm_table(output_folder,tpm_threshold)

# Check if everything looks OK
!ls -al





--2025-02-20 03:04:40--  https://trace.ncbi.nlm.nih.gov/Traces/sra-reads-be/fastq?acc=SRR7083649
Resolving trace.ncbi.nlm.nih.gov (trace.ncbi.nlm.nih.gov)... 130.14.29.113, 2607:f220:41e:4290::113
Connecting to trace.ncbi.nlm.nih.gov (trace.ncbi.nlm.nih.gov)|130.14.29.113|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: unspecified [application/x-gzip]
Saving to: ‘SRR7083649.fastq.gz’

SRR7083649.fastq.gz     [        <=>         ]   2.25M  1.30MB/s    in 1.7s    

2025-02-20 03:04:44 (1.30 MB/s) - ‘SRR7083649.fastq.gz’ saved [2362513]


[quant] fragment length distribution is truncated gaussian with mean = 75, sd = 5
[index] k-mer length: 31
[index] number of targets: 207,133
[index] number of k-mers: 116,708,646
[index] number of equivalence classes: 849,459
[quant] running in single-end mode
[quant] will process file 1: SRR7083649.fastq.gz
[quant] finding pseudoalignments for the reads ... done
[quant] processed 105,452 reads, 76,532 reads pseudoaligned
[ 

In [ ]:
# To inspect results for one of the samples change the names/paths below
!ls -al output_SRR7083635
!head -10 output_SRR7083635/gene_tpm.tsv
!grep "ESR" output_SRR7083635/gene_tpm.tsv

total 6304
drwxr-xr-x 2 root root    4096 Feb 20 02:54 .
drwxr-xr-x 1 root root    4096 Feb 20 03:05 ..
-rw-r--r-- 1 root root 6435071 Feb 20 03:01 abundance.tsv
-rw-r--r-- 1 root root    1199 Feb 20 03:01 gene_tpm.tsv
-rw-r--r-- 1 root root     379 Feb 20 03:01 run_info.json
PSMC4 9121.64
SLC39A6 2276.48
MYBL2 12776.1
BCL2 568.642
APOBEC3G 12388
UBE2C 44123.6
NDC80 6404.35
GZMK 14070.6
AURKA 1772.79
BLVRA 3225.32
ESR1 164.18


What are the units here and how are they defined? For in-depth analysis read this: https://translational-medicine.biomedcentral.com/articles/10.1186/s12967-021-02936-w

How do we even know if these numbers make sense. Consider the fact that LumA and Basal subtypes are expected to be very different. How could one test that? What about PAM50 genes (if they are included in the panel - remember, this is a targeted profiling)? What about clustering and separability?

What is the effect of changing the parameters here, including kmer length, expected fragment length and its variability? What about the fact that we assumed these are not paired end reads, providing only one fastq file at a time?

Part B: With the questions posed above, consider a simple 1-gene classifier of LumA vs. Basal breast tumors. Recall that LumA is ER positive whereas generally speaking Basal tumors are ER negative. While this refers to protein abundance, mRNA levels for ESR1 gene are often sufficient to discriminate between these two subtypes.

Compute the mean expression for ESR1 in both groups, as well as standard deviations in each group. Report those values and based on that suggest a threshold that can be used for classification. For extra points, how would you extend this concept to PAM50 50 gene panel?